## Setup and Imports

In [1]:
import os
import json
import numpy as np
import torch
from pathlib import Path
from collections import Counter

from tqdm import tqdm
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

from seqeval.metrics import precision_score, recall_score, f1_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


###  Define label space (entity types + BIO tagging)
 definition of the 13 GutBrainIE entity categories and expansions into BIO tags:
 - "O" for tokens outside any entity
 - "B-<label>" for the first token of an entity mention
 - "I-<label>" for continuation tokens
 Then we build `label2id` / `id2label` mappings so the model can train and decode labels.

 Finally we set:
 - the pretrained backbone (BioBERT)
 - the output directory where the fine-tuned model will be saved.

In [2]:
# Define entity labels
ENTITY_LABELS = [
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "DDF",
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique"
]

# Create BIO tags for each entity label
label_list = ['O']  # Outside
for entity_label in ENTITY_LABELS:
    label_list.append(f'B-{entity_label}')  # Beginning
    label_list.append(f'I-{entity_label}')  # Inside

label2id = {k: v for v, k in enumerate(label_list)}
id2label = {v: k for v, k in enumerate(label_list)}

print(f"Total labels: {len(label_list)}")
print(f"\nFirst 10 labels: {label_list[:10]}")

# Model configuration
model_name = "dmis-lab/biobert-v1.1"  # BioBERT for biomedical text
output_model_dir = "models/bert_biomedbert_ner_twopass_annotator_weights_bronze"

print(f"\nModel: {model_name}")
print(f"Output directory: {output_model_dir}")

Total labels: 27

First 10 labels: ['O', 'B-anatomical location', 'I-anatomical location', 'B-animal', 'I-animal', 'B-bacteria', 'I-bacteria', 'B-biomedical technique', 'I-biomedical technique', 'B-chemical']

Model: dmis-lab/biobert-v1.1
Output directory: models/bert_biomedbert_ner_twopass_annotator_weights


## Data Loading Functions
This section defines two helper functions:
 - `load_ner_data`: loads multiple JSON annotation files and merges them into a single dictionary keyed by PMID.
 - `prepare_documents_for_ner`: splits each article into two separate training examples:  one for the title and one for the abstract. This is important because entities are
  annotated with a `location` field (title/abstract) and the spans are relative to that text segment.

In [3]:
ANNOTATOR_WEIGHTS = {
    "expert":     1.0,
    "student_A":  0.75,
    "student_B":  0.55,
    "distant":    0.20,   # ← copre sia bronze che distant esistenti
    "silver_2025": 0.40,
}
DEFAULT_ANN_WEIGHT = 1

def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def normalize_annotator(a: str) -> str:
    a = (a or "").strip()
    # se nel json trovi varianti tipo "student a" o "Student_A"
    a_low = a.lower()
    if a_low in {"student a", "studenta", "student_a"}:
        return "student_A"
    if a_low in {"student b", "studentb", "student_b"}:
        return "student_B"
    if a_low in {"expert"}:
        return "expert"
    if a_low in {"distant"}:
        return "distant"
    if a_low in {"silver_2025"}:
        return "silver_2025"
    return a  # fallback

def load_train_with_priority(
    gold_path: Path,
    silver_path: Path,
    silver_2025_path: Path | None = None,
    bronze_path=None,
):
    """
    Regola:
    - gold vince sempre se stesso PMID è presente anche altrove
    - silver_2025 e silver entrano solo per PMIDs non in gold
    - per silver_2025 forziamo annotator="silver_2025" (peso 0.4)
    """
    gold = load_json(gold_path)
    silver = load_json(silver_path)

    merged = dict(gold)  # gold first
    print("Loaded GOLD:", len(gold))
    print("Loaded SILVER:", len(silver))

    # add silver docs not in gold
    add_silver = 0
    for pmid, art in silver.items():
        if pmid not in merged:
            merged[pmid] = art
            add_silver += 1
    print("Added SILVER (not in gold):", add_silver)

    if silver_2025_path is not None and silver_2025_path.exists():
        silver25 = load_json(silver_2025_path)
        print("Loaded SILVER_2025:", len(silver25))
        add_silver25 = 0
        for pmid, art in silver25.items():
            if pmid in merged:
                continue
            meta = art.get("metadata", {}) or {}
            meta["annotator"] = "silver_2025"
            art["metadata"] = meta
            merged[pmid] = art
            add_silver25 += 1
        print("Added SILVER_2025 (not in gold/silver):", add_silver25)
     # ← AGGIUNGI QUESTO BLOCCO
    if bronze_path is not None and Path(bronze_path).exists():
        bronze = load_json(Path(bronze_path))
        print("Loaded BRONZE:", len(bronze))
        add_bronze = 0
        for pmid, art in bronze.items():
            if pmid in merged:
                continue
            meta = art.get("metadata", {}) or {}
            meta["annotator"] = "distant"   # stesso peso di distant=0.20
            art["metadata"] = meta
            merged[pmid] = art
            add_bronze += 1
        print("Added BRONZE (not in gold/silver/silver_2025):", add_bronze)
    print("TOTAL merged PMIDs:", len(merged))
    return merged

In [4]:
from pathlib import Path

def prepare_documents_for_ner(data):
    docs = []
    for pmid, article in data.items():
        meta = article.get("metadata", {}) or {}
        ents = article.get("entities", []) or []

        title = (meta.get("title") or "").strip()
        abstract = (meta.get("abstract") or "").strip()

        annotator = normalize_annotator(meta.get("annotator"))

        if title:
            docs.append({
                "pmid": str(pmid),
                "location": "title",
                "text": title,
                "entities": [e for e in ents if e.get("location") == "title"],
                "annotator": annotator,
            })
        if abstract:
            docs.append({
                "pmid": str(pmid),
                "location": "abstract",
                "text": abstract,
                "entities": [e for e in ents if e.get("location") == "abstract"],
                "annotator": annotator,
            })
    return docs


def align_labels_with_tokens(text, entities, tokenizer, label2id, max_length=512):
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        return_special_tokens_mask=True,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
    )

    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]
    offset_mapping = encoding["offset_mapping"]
    special_mask = encoding["special_tokens_mask"]

    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    labels = ["O"] * len(input_ids)

    sorted_entities = sorted(
        entities,
        key=lambda e: (int(e["start_idx"]), -(int(e["end_idx"]) - int(e["start_idx"]))),
    )

    labeled_positions = set()

    for ent in sorted_entities:
        ent_start = int(ent["start_idx"])
        ent_end_excl = int(ent["end_idx"]) + 1
        ent_label = str(ent["label"])

        ent_token_start = None
        ent_token_end = None

        for idx, ((tok_start, tok_end), is_special) in enumerate(zip(offset_mapping, special_mask)):
            if is_special == 1:
                continue
            tok_start = int(tok_start); tok_end = int(tok_end)
            if tok_end <= tok_start:
                continue
            if tok_start < ent_end_excl and tok_end > ent_start:
                if ent_token_start is None:
                    ent_token_start = idx
                ent_token_end = idx

        if ent_token_start is not None and ent_token_end is not None:
            for i in range(ent_token_start, ent_token_end + 1):
                if i in labeled_positions:
                    continue
                tag = f"B-{ent_label}" if i == ent_token_start else f"I-{ent_label}"
                if tag in label2id:
                    labels[i] = tag
                    labeled_positions.add(i)

    label_ids = [label2id.get(tag, label2id["O"]) for tag in labels]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": label_ids,
        "tokens": tokens,
        "special_tokens_mask": special_mask,
    }


def build_processed(docs, tokenizer, label2id):
    out = []
    for d in tqdm(docs, desc="Align BIO"):
        ex = align_labels_with_tokens(d["text"], d["entities"], tokenizer, label2id, max_length=512)
        ann = d.get("annotator") or ""
        ex["sample_weight"] = float(ANNOTATOR_WEIGHTS.get(ann, DEFAULT_ANN_WEIGHT))
        out.append(ex)
    return out


## Load Training and Dev Data
loading of the training data (gold/platinum/silver) and development data from the provided dev split. Then it converts articles into per-segment examples (title + abstract), producing `train_documents` and `dev_documents`.

In [5]:
import json
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """
    Risale le cartelle finché trova una directory che contiene 'data'.
    Funziona anche se il notebook parte da src/ner/...
    """
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "data").exists():
            return p
    raise FileNotFoundError(f"Non trovo la cartella 'data' risalendo da: {start}")

PROJECT_ROOT = find_repo_root(Path.cwd())
DATA_ROOT = PROJECT_ROOT / "data" / "GutBrainIE_Full_Collection_2026" / "Annotations"
TRAIN_GOLD = DATA_ROOT / "Train" / "gold_quality" / "json_format" / "train_gold.json"
TRAIN_SILVER = DATA_ROOT / "Train" / "silver_quality" / "json_format" / "train_silver.json"
TRAIN_SILVER_2025 = DATA_ROOT / "Train" / "silver_quality" / "json_format" / "train_silver_2025.json"
TRAIN_BRONZE = DATA_ROOT / "Train" / "bronze_quality" / "json_format" / "train_bronze.json"
DEV_PATH = DATA_ROOT / "Dev" / "json_format" / "dev.json"

# tokenizer + model
model_name = "dmis-lab/biobert-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_data = load_train_with_priority(TRAIN_GOLD, TRAIN_SILVER, TRAIN_SILVER_2025, TRAIN_BRONZE)
dev_data = load_json(DEV_PATH)

train_docs = prepare_documents_for_ner(train_data)
dev_docs = prepare_documents_for_ner(dev_data)

processed_train = build_processed(train_docs, tokenizer, label2id)
processed_dev = build_processed(dev_docs, tokenizer, label2id)

# DEV weight neutro
for ex in processed_dev:
    ex["sample_weight"] = 1.0

print("Train segments:", len(processed_train))
print("Dev segments:", len(processed_dev))

# sanity: quanti pesi diversi
wcnt = Counter([ex["sample_weight"] for ex in processed_train])
print("Train sample_weight distribution:", dict(wcnt))

Loaded GOLD: 639
Loaded SILVER: 811
Added SILVER (not in gold): 811
Loaded SILVER_2025: 499
Added SILVER_2025 (not in gold/silver): 499
TOTAL merged PMIDs: 1949


Align BIO: 100%|██████████| 160/160 [00:00<00:00, 490.38it/s]

Train segments: 3898
Dev segments: 160
Train sample_weight distribution: {1.0: 1278, 0.75: 986, 0.55: 636, 0.4: 998}


In [6]:
from collections import Counter
c = Counter((d.get("annotator") or "<MISSING>").strip() for d in train_docs)
print(c.most_common(20))

[('silver_2025', 998), ('student_A', 986), ('student_B', 636), ('expert_4', 196), ('expert_6', 140), ('expert_3', 138), ('expert_1', 136), ('expert_5', 134), ('expert_2', 126), ('expert_7', 108), ('expert_10', 50), ('expert_9', 50), ('expert_11', 50), ('expert_13', 50), ('expert_8', 50), ('expert_12', 50)]


## Initialize BERT Model and Tokenizer
This cell loads
- the BioBERT tokenizer
 - the BioBERT model with a token-classification head sized to our BIO label space

In [7]:
# Initialize tokenizer and model
print("Initializing BERT tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

print(f"✓ Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"✓ Model loaded with {model.num_labels} labels")

# Test tokenization
sample_text = "The gut microbiome plays a role in Parkinson's disease."
tokens = tokenizer.tokenize(sample_text)
print(f"\nSample tokenization: {tokens}")

Initializing BERT tokenizer and model...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 564.27it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]              
BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Tokenizer loaded: BertTokenizer
✓ Model loaded with 27 labels

Sample tokenization: ['The', 'gut', 'micro', '##bio', '##me', 'plays', 'a', 'role', 'in', 'Parkinson', "'", 's', 'disease', '.']


## BIO Tag Generation for Training Data

The dataset provides entity spans as character offsets in the raw text segment.
 BERT training, however, needs one label per token. This function performs that alignment:

**Key design choices:**
 1) We use `return_offsets_mapping=True` to get (start_char, end_char) for each token.
 2) We initialize all labels to "O".
 3) We sort entities by (start position, longer span first) to make overlap handling deterministic:
    if two entities overlap, the longer one is applied first.
 4) We mark tokens as part of an entity if their offset range overlaps the entity character span.
 5) We prevent double-labeling with `labeled_positions` (first entity wins).

 Important detail for GutBrainIE:
 - `end_idx` in the dataset is inclusive, while Hugging Face offsets are exclusive.
   We convert to exclusive end by using `end_idx + 1` before overlap checks.

In [8]:
def align_labels_with_tokens(text, entities, tokenizer, label2id, max_length=512):
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        return_special_tokens_mask=True,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
    )

    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]
    offset_mapping = encoding["offset_mapping"]          # (start,end) end exclusive
    special_mask = encoding["special_tokens_mask"]       # 1 if special token

    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    labels = ["O"] * len(input_ids)

    sorted_entities = sorted(
        entities,
        key=lambda e: (int(e["start_idx"]), -(int(e["end_idx"]) - int(e["start_idx"]))),
    )

    labeled_positions = set()

    for ent in sorted_entities:
        ent_start = int(ent["start_idx"])
        ent_end_excl = int(ent["end_idx"]) + 1  # dataset end inclusive -> exclusive
        ent_label = str(ent["label"])

        ent_token_start = None
        ent_token_end = None

        for idx, ((tok_start, tok_end), is_special) in enumerate(zip(offset_mapping, special_mask)):
            if is_special == 1:
                continue
            tok_start = int(tok_start); tok_end = int(tok_end)
            if tok_end <= tok_start:
                continue

            if tok_start < ent_end_excl and tok_end > ent_start:
                if ent_token_start is None:
                    ent_token_start = idx
                ent_token_end = idx

        if ent_token_start is not None and ent_token_end is not None:
            for i in range(ent_token_start, ent_token_end + 1):
                if i in labeled_positions:
                    continue
                tag = f"B-{ent_label}" if i == ent_token_start else f"I-{ent_label}"
                if tag in label2id:
                    labels[i] = tag
                    labeled_positions.add(i)

    label_ids = [label2id.get(tag, label2id["O"]) for tag in labels]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": label_ids,
        "tokens": tokens,
    }

## Process Training and Dev Data with BIO Tags
This step applies the BIO alignment to every title/abstract segment

## Prepare Dataset for BERT Training

In [9]:
class NERDataset(Dataset):
    def __init__(self, processed):
        self.data = processed

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        return {
            "input_ids": torch.tensor(x["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(x["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(x["labels"], dtype=torch.long),
            "sample_weight": float(x.get("sample_weight", 1.0)),
        }

train_dataset = NERDataset(processed_train)
dev_dataset = NERDataset(processed_dev)

class WeightedTokenCollator:
    def __init__(self, tokenizer):
        self.base = DataCollatorForTokenClassification(
            tokenizer=tokenizer,
            padding=True,
            return_tensors="pt",
            label_pad_token_id=-100,
        )

    def __call__(self, features):
        weights = torch.tensor([f["sample_weight"] for f in features], dtype=torch.float)
        for f in features:
            f.pop("sample_weight", None)
        batch = self.base(features)
        batch["sample_weight"] = weights
        return batch

data_collator = WeightedTokenCollator(tokenizer)

In [10]:
# Create datasets
print("Creating training datasets...")

train_dataset = NERDataset(processed_train)
dev_dataset = NERDataset(processed_dev)

print(f"✓ Training dataset: {len(train_dataset)} examples")
print(f"✓ Dev dataset: {len(dev_dataset)} examples")

Creating training datasets...
✓ Training dataset: 3898 examples
✓ Dev dataset: 160 examples


### Define Evaluation metric (seqeval over BIO tags)
 This cell defines a `compute_metrics_seqeval` function for Hugging Face Trainer.
 It converts model outputs (logits) into predicted BIO tags and compares them to gold BIO tags using seqeval.

**Implementation details:**
 - `argmax` selects the most likely tag per token.
 - tokens with label -100 are skipped (ignored padding/special positions).
 - seqeval computes precision/recall/F1 at the entity level from BIO sequences.

In [11]:
import torch
from seqeval.metrics import precision_score, recall_score, f1_score

def compute_metrics_seqeval(p):
    logits, labels = p
    preds = np.argmax(logits, axis=-1)

    true_labels = []
    true_preds = []

    for pred_seq, label_seq in zip(preds, labels):
        seq_true = []
        seq_pred = []
        for p_id, l_id in zip(pred_seq, label_seq):
            if l_id == -100:
                continue
            seq_true.append(id2label[int(l_id)])
            seq_pred.append(id2label[int(p_id)])
        true_labels.append(seq_true)
        true_preds.append(seq_pred)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }


##  Training hyperparameters (stability + stronger convergence)
 This cell sets the key training choices:
 - **learning_rate = 3e-5** with **warmup_ratio=0.1** for stability
 - **gradient_accumulation_steps=2** to increase effective batch size without extra GPU memory
- **num_train_epochs=5** to allow the NER head to converge better than short runs
 - best model selection based on seqeval F1
 - fp16 enabled if CUDA is available

In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=output_model_dir,

    # Core optimization
    learning_rate=3e-5,                 # often better than 2e-5 for BioBERT NER
    lr_scheduler_type="linear",
    warmup_ratio=0.1,                   # critical for stability with higher LR
    weight_decay=0.01,

    # Batch/effective batch
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,      # effective batch = 16 (usually helps)

    # Training length
    num_train_epochs=5,                 # 3 is often too short for NER

    # Evaluation / checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    label_smoothing_factor=0.0,  # keep 0 for token classification; don't smooth rare labels away
    # If you have compute_metrics with seqeval later, use f1
    metric_for_best_model="f1", #CHANGED
    greater_is_better=True,

    # Runtime / logging
    logging_steps=100,
    save_total_limit=2,
    seed=42,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
)

print("✓ Training configuration ready")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✓ Training configuration ready
  Batch size: 8
  Epochs: 5
  Learning rate: 3e-05


## Train BERT Model


## Class-weighted loss (handle label imbalance)
 Many GutBrainIE labels are rare (e.g., some entity types appear much less than "O").
 This cell:
 1) Counts token-level label frequencies in the training set.
 2) Builds class weights using inverse-frequency^power (power=0.5 => sqrt inverse frequency).
 3) Normalizes weights to mean=1 and clips them to avoid extreme gradients

Then it defines a custom Trainer that overrides `compute_loss` to use:
  CrossEntropyLoss(weight=class_weights, ignore_index=-100)
 This generally improves recall for under-represented labels without exploding training.

In [13]:
import torch
from collections import Counter
from transformers import Trainer

def compute_class_weights(processed, num_labels, ignore_index=-100, power=0.5):
    counts = Counter()
    for ex in processed:
        for y in ex["labels"]:
            if int(y) == ignore_index:
                continue
            counts[int(y)] += 1
    freqs = np.zeros(num_labels, dtype=np.float64)
    for c in range(num_labels):
        freqs[c] = counts.get(c, 0)
    freqs[freqs == 0] = 1.0
    weights = (1.0 / freqs) ** power
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float)

class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        sample_weight = inputs.get("sample_weight", None)

        model_inputs = {k: v for k, v in inputs.items() if k not in {"labels", "sample_weight"}}
        outputs = model(**model_inputs)
        logits = outputs.logits  # (B, T, C)

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
            ignore_index=-100,
            reduction="none",
        )

        B, T, C = logits.shape
        token_loss = loss_fct(logits.view(-1, C), labels.view(-1)).view(B, T)

        mask = (labels != -100).float()
        token_loss = token_loss * mask
        denom = mask.sum(dim=1).clamp(min=1.0)
        sample_loss = token_loss.sum(dim=1) / denom

        if sample_weight is not None:
            sample_weight = sample_weight.to(sample_loss.device)
            sample_loss = sample_loss * sample_weight

        loss = sample_loss.mean()
        return (loss, outputs) if return_outputs else loss


### Initialize Trainer (training loop + evaluation)

### Train the model

In [14]:
def compute_metrics_seqeval(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_labels, true_preds = [], []
    for pred_seq, label_seq in zip(preds, labels):
        seq_true, seq_pred = [], []
        for p_id, l_id in zip(pred_seq, label_seq):
            if l_id == -100:
                continue
            seq_true.append(id2label[int(l_id)])
            seq_pred.append(id2label[int(p_id)])
        true_labels.append(seq_true)
        true_preds.append(seq_pred)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
).to(device)

class_weights = compute_class_weights(processed_train, num_labels=len(label_list), power=0.5)
class_weights = torch.clamp(class_weights, min=0.5, max=5.0)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics_seqeval,
    class_weights=class_weights,
)

trainer.train()
trainer.save_model(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)
print("Saved to:", training_args.output_dir)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 517.30it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]              
BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.713693,0.334038,0.661596,0.737081,0.697302
2,0.438121,0.300288,0.688335,0.787929,0.734773
3,0.322614,0.287222,0.730640,0.807358,0.767086
4,0.251340,0.290944,0.704772,0.818107,0.757222
5,0.214054,0.299346,0.725612,0.821000,0.770365


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.75it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

Saved to: models/bert_biomedbert_ner_twopass_annotator_weights


In [15]:
from collections import Counter
ann_cnt = Counter([d["annotator"] for d in train_docs])
print("Annotators in train_docs:", ann_cnt)

w_cnt = Counter([ex["sample_weight"] for ex in processed_train])
print("Weights in processed_train:", w_cnt)

Annotators in train_docs: Counter({'silver_2025': 998, 'student_A': 986, 'student_B': 636, 'expert_4': 196, 'expert_6': 140, 'expert_3': 138, 'expert_1': 136, 'expert_5': 134, 'expert_2': 126, 'expert_7': 108, 'expert_10': 50, 'expert_9': 50, 'expert_11': 50, 'expert_13': 50, 'expert_8': 50, 'expert_12': 50})
Weights in processed_train: Counter({1.0: 1278, 0.4: 998, 0.75: 986, 0.55: 636})


## Save Trained Model

In [16]:
# Save the trained model
print("Saving trained model...")

os.makedirs(output_model_dir, exist_ok=True)
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

print(f"✓ Model saved to: {output_model_dir}")

Saving trained model...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.12it/s]

✓ Model saved to: models/bert_biomedbert_ner_twopass_annotator_weights
